
# Election Bloc Change Prediction Project  
## Notebook 00 — Clean project setup

This notebook starts the project again from a clean slate.

### Reset rule

The only trusted inputs are the files under:

```text
data/raw/
```

All previous derived files are ignored and deleted from the local Colab copy:

```text
data/interim/
data/processed/
reports/tables/
reports/figures/
reports/summaries/
models/
```

The notebook then:

1. locates or clones the GitHub repository;
2. recreates the complete project folder structure;
3. inventories every raw input file;
4. checks the expected election and socioeconomic source files;
5. writes a new project configuration and setup audit;
6. creates a downloadable ZIP of Notebook 00 outputs.

No model data are created in this notebook.


In [ ]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
import os
import shutil
import subprocess
import sys
import time

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO_ROOT = Path("/content/Election_Bloc_Prediction_Project")

MODELED_BLOCS = [
    "Right",
    "Center_Left",
    "Haredi",
    "Arab",
]

ELECTION_NUMBERS = list(range(16, 26))
FINAL_TEST_TRANSITION = "K24_to_K25"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)


## 1. Locate or clone the repository

In [ ]:

def locate_or_clone_repository():
    explicit_root = os.getenv("ELECTION_PROJECT_ROOT")
    candidates = []

    if explicit_root:
        candidates.append(Path(explicit_root).expanduser())

    current = Path.cwd().resolve()
    candidates.extend([
        current,
        *current.parents,
        DEFAULT_REPO_ROOT,
    ])

    checked = set()

    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except FileNotFoundError:
            continue

        if candidate in checked:
            continue

        checked.add(candidate)

        if (candidate / "data" / "raw").exists():
            return candidate

    if not Path("/content").exists():
        raise FileNotFoundError(
            "The repository could not be located. "
            "Set ELECTION_PROJECT_ROOT to the repository folder."
        )

    if DEFAULT_REPO_ROOT.exists():
        shutil.rmtree(DEFAULT_REPO_ROOT)

    print("Repository not found locally.")
    print("Cloning the current GitHub repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(DEFAULT_REPO_ROOT),
        ],
        check=True,
    )

    return DEFAULT_REPO_ROOT


REPO_ROOT = locate_or_clone_repository()

if (REPO_ROOT / ".git").exists():
    print("Synchronizing the local copy with GitHub...")
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_ROOT),
            "pull",
            "--ff-only",
            "origin",
            "main",
        ],
        check=True,
    )

print("Repository root:", REPO_ROOT)
print("Raw data root:", REPO_ROOT / "data" / "raw")


## 2. Delete old local outputs and recreate the project structure

In [ ]:

RAW_DIR = REPO_ROOT / "data" / "raw"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

REPORTS_DIR = REPO_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
SUMMARIES_DIR = REPORTS_DIR / "summaries"

MODELS_DIR = REPO_ROOT / "models"
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Raw-data folder not found: {RAW_DIR}"
    )

# This affects only the current local runtime copy.
# It does not delete files from GitHub.
DERIVED_DIRECTORIES = [
    INTERIM_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    SUMMARIES_DIR,
    MODELS_DIR,
]

for directory in DERIVED_DIRECTORIES:
    if directory.exists():
        shutil.rmtree(directory)

DIRECTORIES_TO_CREATE = [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    SUMMARIES_DIR,
    MODELS_DIR,
    NOTEBOOKS_DIR,
]

for directory in DIRECTORIES_TO_CREATE:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

project_structure = pd.DataFrame({
    "folder": [
        str(path.relative_to(REPO_ROOT))
        for path in DIRECTORIES_TO_CREATE
    ],
    "exists": [
        path.exists()
        for path in DIRECTORIES_TO_CREATE
    ],
})

display(project_structure)

if not project_structure["exists"].all():
    raise RuntimeError(
        "At least one required project directory was not created."
    )


## 3. Inventory every raw file

In [ ]:

raw_files = sorted(
    path
    for path in RAW_DIR.rglob("*")
    if path.is_file()
)

if not raw_files:
    raise RuntimeError(
        "The raw-data folder is empty."
    )

inventory_rows = []

for path in raw_files:
    relative_path = path.relative_to(REPO_ROOT)
    relative_to_raw = path.relative_to(RAW_DIR)

    top_folder = (
        relative_to_raw.parts[0]
        if len(relative_to_raw.parts) > 1
        else "(raw root)"
    )

    inventory_rows.append({
        "relative_path": str(relative_path),
        "raw_category": top_folder,
        "filename": path.name,
        "extension": "".join(path.suffixes).lower(),
        "size_bytes": path.stat().st_size,
        "size_mb": round(
            path.stat().st_size / (1024 ** 2),
            4,
        ),
    })

raw_inventory = pd.DataFrame(
    inventory_rows
).sort_values(
    [
        "raw_category",
        "filename",
    ]
).reset_index(
    drop=True
)

display(raw_inventory)

print("\nRaw files by category:")
display(
    raw_inventory
    .groupby("raw_category")
    .agg(
        files=("filename", "count"),
        total_mb=("size_mb", "sum"),
    )
    .sort_index()
)

if (raw_inventory["size_bytes"] <= 0).any():
    empty_files = raw_inventory.loc[
        raw_inventory["size_bytes"] <= 0,
        "relative_path",
    ].tolist()

    raise RuntimeError(
        "Empty raw files were found:\n- "
        + "\n- ".join(empty_files)
    )


## 4. Expected-source audit

In [ ]:

EXPECTED_ELECTION_FILES = {
    16: "data/raw/Election_data/Knesset_16_Election_Results_2003_By_Kalfi.csv.xls",
    17: "data/raw/Election_data/Knesset_17_Election_Results_2006_By_Kalfi.csv.xls",
    18: "data/raw/Election_data/Knesset_18_Election_Results_2009_By_Kalfi.csv.csv",
    19: "data/raw/Election_data/Knesset_19_Election_Results_2013_By_Locality.csv.csv",
    20: "data/raw/Election_data/Knesset_20_Election_Results_2015_By_Locality.csv.csv",
    21: "data/raw/Election_data/Knesset_21_Election_Results_2019_By_Locality.csv",
    22: "data/raw/Election_data/Knesset_22_Election_Results_2019_By_Locality.csv",
    23: "data/raw/Election_data/Knesset_23_Election_Results_2020_By_Locality.csv",
    24: "data/raw/Election_data/Knesset_24_Election_Results_2021_By_Locality.csv",
    25: "data/raw/Election_data/Knesset_25_Election_Results_2022_By_Locality.csv",
}

EXPECTED_SOCIOECONOMIC_FILES = []

for year in [2003, 2006, 2009, 2013, 2015, 2019, 2020, 2021, 2022, 2023]:
    EXPECTED_SOCIOECONOMIC_FILES.append({
        "source": "demographic",
        "year": year,
        "relative_path": (
            f"data/raw/Demographic_data/Demographic_{year}.xlsx"
        ),
    })

for year in [2003, 2006, 2009, 2013, 2015, 2019, 2020, 2021, 2022, 2023]:
    EXPECTED_SOCIOECONOMIC_FILES.append({
        "source": "income",
        "year": year,
        "relative_path": (
            f"data/raw/Average_income/ICBS_{year}.xlsx"
        ),
    })

for year in [2003, 2006, 2009, 2013, 2015, 2019, 2020, 2021, 2022, 2023]:
    EXPECTED_SOCIOECONOMIC_FILES.append({
        "source": "unemployment",
        "year": year,
        "relative_path": (
            f"data/raw/Unemployment_data/Unemployment_{year}.xlsx"
        ),
    })

for year in [2016, 2019, 2020, 2021, 2022, 2023]:
    EXPECTED_SOCIOECONOMIC_FILES.append({
        "source": "education",
        "year": year,
        "relative_path": (
            f"data/raw/Education_data/Education_{year}.xlsx"
        ),
    })

source_audit_rows = []

for election_number, relative_path in EXPECTED_ELECTION_FILES.items():
    path = REPO_ROOT / relative_path

    source_audit_rows.append({
        "source_group": "election",
        "source": f"Knesset_{election_number}",
        "year": None,
        "relative_path": relative_path,
        "required_for_notebook_01": True,
        "exists": path.exists(),
        "size_bytes": (
            path.stat().st_size
            if path.exists()
            else 0
        ),
    })

for item in EXPECTED_SOCIOECONOMIC_FILES:
    path = REPO_ROOT / item["relative_path"]

    source_audit_rows.append({
        "source_group": "socioeconomic",
        "source": item["source"],
        "year": item["year"],
        "relative_path": item["relative_path"],
        "required_for_notebook_01": False,
        "exists": path.exists(),
        "size_bytes": (
            path.stat().st_size
            if path.exists()
            else 0
        ),
    })

expected_source_audit = pd.DataFrame(
    source_audit_rows
)

display(expected_source_audit)

missing_election_files = expected_source_audit.loc[
    expected_source_audit[
        "required_for_notebook_01"
    ]
    & ~expected_source_audit["exists"],
    "relative_path",
].tolist()

if missing_election_files:
    raise FileNotFoundError(
        "Required election files are missing:\n- "
        + "\n- ".join(missing_election_files)
    )

print(
    "All 10 required election files are present."
)

print("\nSocioeconomic coverage by source:")
display(
    expected_source_audit.loc[
        expected_source_audit[
            "source_group"
        ].eq("socioeconomic")
    ]
    .groupby("source")
    .agg(
        expected_files=("relative_path", "count"),
        available_files=("exists", "sum"),
    )
    .sort_index()
)


## 5. Save Notebook 00 outputs

In [ ]:

PROJECT_CONFIG_PATH = (
    SUMMARIES_DIR
    / "project_config.json"
)

RAW_INVENTORY_PATH = (
    TABLES_DIR
    / "raw_file_inventory.csv"
)

EXPECTED_SOURCE_AUDIT_PATH = (
    TABLES_DIR
    / "expected_source_audit.csv"
)

PROJECT_STRUCTURE_PATH = (
    TABLES_DIR
    / "project_structure.csv"
)

NOTEBOOK_SUMMARY_PATH = (
    SUMMARIES_DIR
    / "notebook_00_summary.json"
)

project_config = {
    "project_name": (
        "Election Bloc Change Prediction Project"
    ),
    "repository_url": REPO_URL,
    "modeled_blocs": MODELED_BLOCS,
    "election_numbers": ELECTION_NUMBERS,
    "development_transitions": [
        f"K{number}_to_K{number + 1}"
        for number in range(16, 24)
    ],
    "final_test_transition": (
        FINAL_TEST_TRANSITION
    ),
    "trusted_input_root": "data/raw",
    "canonical_directories": {
        "raw": "data/raw",
        "interim": "data/interim",
        "processed": "data/processed",
        "tables": "reports/tables",
        "figures": "reports/figures",
        "summaries": "reports/summaries",
        "models": "models",
        "notebooks": "notebooks",
    },
    "reset_policy": (
        "Only raw files are trusted. "
        "All derived local outputs are recreated."
    ),
}

PROJECT_CONFIG_PATH.write_text(
    json.dumps(
        project_config,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

raw_inventory.to_csv(
    RAW_INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

expected_source_audit.to_csv(
    EXPECTED_SOURCE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

project_structure.to_csv(
    PROJECT_STRUCTURE_PATH,
    index=False,
    encoding="utf-8-sig",
)

notebook_summary = {
    "notebook": "00_clean_project_setup",
    "created_at_utc": time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime(),
    ),
    "repository_root": str(REPO_ROOT),
    "raw_file_count": int(
        len(raw_inventory)
    ),
    "raw_categories": sorted(
        raw_inventory[
            "raw_category"
        ].unique().tolist()
    ),
    "all_required_election_files_present": (
        len(missing_election_files) == 0
    ),
    "derived_directories_recreated": [
        str(path.relative_to(REPO_ROOT))
        for path in DERIVED_DIRECTORIES
    ],
    "outputs": {
        "project_config": str(
            PROJECT_CONFIG_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "raw_inventory": str(
            RAW_INVENTORY_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "expected_source_audit": str(
            EXPECTED_SOURCE_AUDIT_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "project_structure": str(
            PROJECT_STRUCTURE_PATH.relative_to(
                REPO_ROOT
            )
        ),
    },
}

NOTEBOOK_SUMMARY_PATH.write_text(
    json.dumps(
        notebook_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

OUTPUT_FILES = [
    PROJECT_CONFIG_PATH,
    RAW_INVENTORY_PATH,
    EXPECTED_SOURCE_AUDIT_PATH,
    PROJECT_STRUCTURE_PATH,
    NOTEBOOK_SUMMARY_PATH,
]

output_audit = pd.DataFrame([
    {
        "relative_path": str(
            path.relative_to(
                REPO_ROOT
            )
        ),
        "exists": path.exists(),
        "size_bytes": (
            path.stat().st_size
            if path.exists()
            else 0
        ),
    }
    for path in OUTPUT_FILES
])

display(output_audit)

if (
    (~output_audit["exists"]).any()
    or (output_audit["size_bytes"] <= 0).any()
):
    raise RuntimeError(
        "Notebook 00 did not save all expected outputs."
    )


## 6. Download Notebook 00 outputs

In [ ]:

ZIP_PATH = Path(
    "/content/notebook_00_outputs.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with ZipFile(
    ZIP_PATH,
    mode="w",
    compression=ZIP_DEFLATED,
) as zip_file:
    for path in OUTPUT_FILES:
        zip_file.write(
            path,
            arcname=str(
                path.relative_to(
                    REPO_ROOT
                )
            ),
        )

with ZipFile(
    ZIP_PATH,
    mode="r",
) as zip_file:
    zipped_files = zip_file.namelist()

print("ZIP:", ZIP_PATH)
print("ZIP size:", ZIP_PATH.stat().st_size, "bytes")
print("\nFiles inside the ZIP:")

for filename in zipped_files:
    print("-", filename)

if len(zipped_files) != len(OUTPUT_FILES):
    raise RuntimeError(
        "The ZIP does not contain every Notebook 00 output."
    )

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print(
        "\nNot running in Google Colab. "
        "The ZIP remains at:",
        ZIP_PATH,
    )



## Completion criteria

Notebook 00 is complete only when:

- all ten election files are reported as present;
- the project folders were recreated successfully;
- every raw file has a positive size;
- five Notebook 00 output files appear in the ZIP.

The next notebook is:

```text
01_data_loading_and_party_mapping.ipynb
```

It will read only the raw election files and will recreate the locality-level election table from scratch.
